# Bento 3D — Robô Mediador Social entre Idoso e Família (Brasil)

**Autores:** Matheus Henrique & Lucas Denner  
**Referência Científica:** Noguchi, Y., Kamide, H., & Tanaka, F. (2023). *How Should a Social Mediator Robot Convey Messages About the Self-Disclosures of Elderly People to Recipients?* International Journal of Social Robotics, 15, 1079–1099.

---
### Como executar no Google Colab (GPU T4 15GB VRAM):
1. No menu superior do Colab, verifique se a GPU está ativa em **Ambiente de execução -> Alterar o tipo de ambiente de execução -> GPU T4**.
2. Na **Célula 2**, você pode escolher entre o modelo **Qwen2.5-14B-Instruct (Q4_K_M — ~9.0 GB VRAM, recomendado para a GPU de 15GB do Colab)** com **12K de contexto (12288 tokens)**, ou o modelo **Qwen2.5-7B-Instruct (~4.5 GB VRAM)**.
3. Na última célula, será exibido um link público **`https://....trycloudflare.com`**. Clique nele para abrir o ambiente 3D com microfone e voz ativos.

In [ ]:
# @title 1. Clonar Repositório e Instalar Dependências (Node.js + Cloudflared)
import os

REPO_URL = "https://github.com/Umhenrique/robo-mediador-3d.git"
PROJECT_DIR = "/content/robo-mediador-3d"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

%cd {PROJECT_DIR}
!npm install

# Baixa Cloudflared para criar túnel HTTPS gratuito (necessário para permissão de microfone no navegador)
if not os.path.exists("/usr/local/bin/cloudflared") or os.path.getsize("/usr/local/bin/cloudflared") < 1000000:
    !curl -fL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

print("[OK] Repositório e dependências configurados com sucesso.")

In [ ]:
# @title 2. Escolher e Baixar Modelo LLM (14B ou 7B para 15GB VRAM) + KoboldCpp CUDA + Whisper Small STT
import os

# @markdown Escolha o porte do modelo para rodar nos 15GB de VRAM da GPU T4 do Colab:
MODEL_CHOICE = "Qwen2.5-14B-Instruct-Q4_K_M (~9.0 GB VRAM - Ideal para Colab 15GB)" # @param ["Qwen2.5-14B-Instruct-Q4_K_M (~9.0 GB VRAM - Ideal para Colab 15GB)", "Gemma-2-9B-It-Q5_K_M (~6.4 GB VRAM)", "Qwen2.5-7B-Instruct-Q4_K_M (~4.5 GB VRAM - Mais Rapido)"]
CONTEXT_SIZE = 12288 # @param {type:"integer"}

os.makedirs("/content/models", exist_ok=True)

if "14B" in MODEL_CHOICE:
    MODEL_FILENAME = "Qwen2.5-14B-Instruct-Q4_K_M.gguf"
    MODEL_URL = "https://huggingface.co/bartowski/Qwen2.5-14B-Instruct-GGUF/resolve/main/Qwen2.5-14B-Instruct-Q4_K_M.gguf"
elif "9B" in MODEL_CHOICE:
    MODEL_FILENAME = "gemma-2-9b-it-Q5_K_M.gguf"
    MODEL_URL = "https://huggingface.co/bartowski/gemma-2-9b-it-GGUF/resolve/main/gemma-2-9b-it-Q5_K_M.gguf"
else:
    MODEL_FILENAME = "Qwen2.5-7B-Instruct-Q4_K_M.gguf"
    MODEL_URL = "https://huggingface.co/bartowski/Qwen2.5-7B-Instruct-GGUF/resolve/main/Qwen2.5-7B-Instruct-Q4_K_M.gguf"

KOBOLD_BIN = "/content/koboldcpp-linux-x64"
LLM_MODEL = f"/content/models/{MODEL_FILENAME}"
WHISPER_MODEL = "/content/models/ggml-small-q5_1.bin"

# Salva configuracao escolhida para a Celula 3
with open("/content/selected_model.txt", "w") as f:
    f.write(f"{LLM_MODEL}\n{CONTEXT_SIZE}")

if not os.path.exists(KOBOLD_BIN) or os.path.getsize(KOBOLD_BIN) < 10000000:
    print("Baixando KoboldCpp Linux x64 com suporte a GPU CUDA...")
    !curl -fL https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64 -o {KOBOLD_BIN}
    !chmod +x {KOBOLD_BIN}

if not os.path.exists(LLM_MODEL) or os.path.getsize(LLM_MODEL) < 100000000:
    print(f"Baixando LLM {MODEL_FILENAME}...")
    !curl -fL {MODEL_URL} -o {LLM_MODEL}

if not os.path.exists(WHISPER_MODEL) or os.path.getsize(WHISPER_MODEL) < 10000000:
    print("Baixando Whisper Small (Q5_1) para reconhecimento de voz em Português...")
    !curl -fL https://huggingface.co/ggerganov/whisper.cpp/resolve/main/ggml-small-q5_1.bin -o {WHISPER_MODEL}

print(f"[OK] Modelo selecionado: {LLM_MODEL} | Contexto: {CONTEXT_SIZE} tokens.")

In [ ]:
# @title 3. Iniciar LLM (12K Contexto) na GPU + Servidor Bento 3D + Link HTTPS Público
import os
import subprocess
import time
import re
import urllib.request

# Mata processos anteriores caso a celula seja reexecutada
!pkill -f koboldcpp 2>/dev/null || true
!pkill -f "node src/server.js" 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true

KOBOLD_BIN = "/content/koboldcpp-linux-x64"
if not os.path.exists(KOBOLD_BIN) or os.path.getsize(KOBOLD_BIN) < 10000000:
    print("Baixando binário koboldcpp-linux-x64...")
    !curl -fL https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64 -o {KOBOLD_BIN}
    !chmod +x {KOBOLD_BIN}

LLM_MODEL = "/content/models/Qwen2.5-14B-Instruct-Q4_K_M.gguf"
CONTEXT_SIZE = "12288"
if os.path.exists("/content/selected_model.txt"):
    lines = open("/content/selected_model.txt").read().strip().splitlines()
    if len(lines) >= 2:
        LLM_MODEL, CONTEXT_SIZE = lines[0], lines[1]
elif os.path.exists("/content/models/Qwen2.5-7B-Instruct-Q4_K_M.gguf") and not os.path.exists(LLM_MODEL):
    LLM_MODEL = "/content/models/Qwen2.5-7B-Instruct-Q4_K_M.gguf"

# 1. Inicia o KoboldCpp em background na porta 5001 com Contexto de 12K (12288) e KV Cache Quantizado (Q8)
print(f"Iniciando KoboldCpp na GPU com modelo {os.path.basename(LLM_MODEL)} e contexto de {CONTEXT_SIZE} tokens...")
kobold_log = open("/content/kobold.log", "w")
kobold_proc = subprocess.Popen([
    KOBOLD_BIN,
    "--model", LLM_MODEL,
    "--whispermodel", "/content/models/ggml-small-q5_1.bin",
    "--contextsize", str(CONTEXT_SIZE),
    "--gpulayers", "99",
    "--usecuda",
    "--flashattention",
    "--quantkv", "1",
    "--blasbatchsize", "512",
    "--port", "5001",
    "--skiplauncher"
], stdout=kobold_log, stderr=subprocess.STDOUT)

# Aguarda API do KoboldCpp ficar online (o primeiro boot extrai o CUDA e aloca os 9GB+ na VRAM)
for i in range(120):
    try:
        with urllib.request.urlopen("http://127.0.0.1:5001/api/v1/model", timeout=2) as r:
            if r.status == 200:
                print(f"[OK] LLM ({os.path.basename(LLM_MODEL)}) + Whisper prontos na porta 5001 (Contexto: {CONTEXT_SIZE}).")
                break
    except Exception:
        time.sleep(2)
else:
    print("Aviso: KoboldCpp ainda está carregando. Últimas linhas de /content/kobold.log:")
    !tail -n 15 /content/kobold.log

# 2. Inicia o servidor Node.js do Bento 3D na porta 3030
print("Iniciando servidor Bento 3D na porta 3030...")
node_log = open("/content/bento.log", "w")
node_proc = subprocess.Popen(
    ["node", "src/server.js"],
    cwd="/content/robo-mediador-3d",
    stdout=node_log,
    stderr=subprocess.STDOUT
)
time.sleep(2)

# 3. Abre o túnel HTTPS com Cloudflared
print("Gerando link HTTPS público (Cloudflare Tunnel)...")
tunnel_log = open("/content/tunnel.log", "w+")
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:3030"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

public_url = None
for _ in range(30):
    time.sleep(1)
    tunnel_log.seek(0)
    content = tunnel_log.read()
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
    if m:
        public_url = m.group(0)
        break

print("\n" + "="*75)
if public_url:
    print("BENTO 3D ONLINE! ACESSE O LINK ABAIXO:")
    print(f"URL: {public_url}")
else:
    print("Aviso: Não foi possível capturar a URL automaticamente. Veja /content/tunnel.log")
print("="*75 + "\n")

# Mantém a célula viva enquanto o servidor estiver rodando
try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    print("Encerrando servidores...")
    tunnel_proc.terminate()
    node_proc.terminate()
    kobold_proc.terminate()